# Pipeline Results Analysis

Analysis of outputs: **archetype cluster assignments**, **soft GMM probabilities**, **LLM cluster labels**, and **KNN player similarities**.

Sections:
1. Load artifacts from S3
2. Cluster overview (sizes + LLM labels)
3. Cluster profiles (feature means heatmap)
4. Soft assignment analysis (confidence distribution)
5. PCA 2D scatter
6. Player lookup (archetype + soft probabilities)
7. Player similarity explorer
8. Distance distributions
9. Cross-cluster neighbor analysis

## 1. Setup

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import s3fs
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", palette="tab10")

BUCKET = "diamond-dna-data-lake"
GOLD_PREFIX = "gold/statcast"
YEAR = 2025
ROLES = ["batter", "pitcher", "catcher"]

fs = s3fs.S3FileSystem()


def s3_path(role: str, filename: str) -> str:
    return f"s3://{BUCKET}/{GOLD_PREFIX}/{role}/year={YEAR}/{filename}"


def read_parquet(role: str, filename: str) -> pd.DataFrame | None:
    path = s3_path(role, filename)
    try:
        with fs.open(path, "rb") as f:
            return pd.read_parquet(f)
    except FileNotFoundError:
        print(f"[missing] {path}")
        return None


def read_json(role: str, filename: str) -> dict | None:
    path = s3_path(role, filename)
    try:
        with fs.open(path, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"[missing] {path}")
        return None

## 2. Load Gold Artifacts

In [ ]:
archetypes: dict[str, pd.DataFrame] = {}
neighbors: dict[str, pd.DataFrame] = {}
cluster_labels: dict[str, dict] = {}
clustering_meta: dict[str, dict] = {}

for role in ROLES:
    archetypes[role] = read_parquet(role, "player_year_archetypes.parquet")
    neighbors[role] = read_parquet(role, "player_year_similar_neighbors.parquet")
    cluster_labels[role] = read_json(role, "cluster_labels.json")
    clustering_meta[role] = read_json(role, "archetype_clustering_metadata.json")

for role in ROLES:
    n = len(archetypes[role]) if archetypes[role] is not None else "N/A"
    nbr = len(neighbors[role]) if neighbors[role] is not None else "N/A"
    k = clustering_meta[role].get("n_clusters", "?") if clustering_meta[role] else "?"
    print(f"{role:8s}  players={n:>4}  neighbor_rows={nbr:>5}  k={k}")

## 3. Cluster Overview

In [ ]:
def label_for(role: str, cluster_id: int) -> str:
    """Return LLM archetype name if available, else 'Cluster N'."""
    labels = cluster_labels.get(role)
    if labels and "labels" in labels:
        entry = labels["labels"].get(str(cluster_id))
        if entry:
            return entry.get("name", f"Cluster {cluster_id}")
    return f"Cluster {cluster_id}"


fig, axes = plt.subplots(1, len(ROLES), figsize=(14, 4), sharey=False)
for ax, role in zip(axes, ROLES):
    df = archetypes.get(role)
    if df is None:
        ax.set_title(f"{role} (no data)")
        continue
    counts = df.groupby("cluster_id").size().sort_index()
    tick_labels = [label_for(role, i) for i in counts.index]
    ax.bar(range(len(counts)), counts.values, color=sns.color_palette("tab10", len(counts)))
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels(tick_labels, rotation=30, ha="right", fontsize=8)
    ax.set_title(f"{role.capitalize()} — {len(df)} players", fontsize=11)
    ax.set_ylabel("Players")
    for i, v in enumerate(counts.values):
        ax.text(i, v + 0.5, str(v), ha="center", va="bottom", fontsize=8)

fig.suptitle(f"Cluster Sizes — {YEAR}", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Print LLM archetype descriptions
for role in ROLES:
    labels = cluster_labels.get(role)
    if not labels:
        print(f"--- {role}: no cluster labels ---\n")
        continue
    print(f"=== {role.upper()} ===  (model: {labels.get('model', '?')})")
    for cid, info in sorted(labels.get("labels", {}).items(), key=lambda x: int(x[0])):
        name = info.get("name", "?")
        desc = info.get("description", "")
        print(f"  {cid}: {name}")
        print(f"     {desc[:120]}..." if len(desc) > 120 else f"     {desc}")
    print()

## 4. Cluster Profiles — Feature Means Heatmap

Z-scored mean of each feature per cluster (rows = features, columns = clusters).

In [ ]:
ID_COLS = {"player_id", "player_name", "year", "role", "n_pitches_total",
           "primary_position", "cluster_id", "cluster_id_secondary",
           "prob_primary", "prob_secondary"}


def feature_cols(df: pd.DataFrame) -> list[str]:
    prob_cols = {c for c in df.columns if c.startswith("prob_")}
    return [c for c in df.select_dtypes(include="number").columns
            if c not in ID_COLS and c not in prob_cols]


for role in ROLES:
    df = archetypes.get(role)
    if df is None:
        continue
    feats = feature_cols(df)
    if not feats:
        continue

    means = df.groupby("cluster_id")[feats].mean()
    # Z-score each feature across clusters for visual contrast
    z = (means - means.mean()) / means.std().replace(0, 1)

    col_labels = {str(i): label_for(role, i) for i in means.index}
    z_display = z.rename(index=col_labels)

    fig, ax = plt.subplots(figsize=(max(6, len(feats) * 0.4), max(3, len(means) * 0.7)))
    sns.heatmap(
        z_display.T,
        ax=ax,
        cmap="RdBu_r",
        center=0,
        annot=False,
        linewidths=0.3,
        cbar_kws={"label": "z-score"},
    )
    ax.set_title(f"{role.capitalize()} Cluster Feature Profiles — {YEAR}", fontsize=12)
    ax.set_xlabel("Feature")
    ax.set_ylabel("Cluster")
    plt.xticks(rotation=45, ha="right", fontsize=7)
    plt.tight_layout()
    plt.show()

## 5. Soft Assignment Analysis

Distribution of `prob_primary` (how confident the GMM was in each assignment).

In [ ]:
fig, axes = plt.subplots(1, len(ROLES), figsize=(14, 4), sharey=False)
for ax, role in zip(axes, ROLES):
    df = archetypes.get(role)
    if df is None or "prob_primary" not in df.columns:
        ax.set_title(f"{role} (no soft probs)")
        continue
    meta = clustering_meta.get(role) or {}
    mean_p = meta.get("mean_max_prob", df["prob_primary"].mean())
    frac_conf = meta.get("frac_confident_p_ge_0_7", (df["prob_primary"] >= 0.7).mean())
    ax.hist(df["prob_primary"], bins=20, color="steelblue", edgecolor="white", alpha=0.85)
    ax.axvline(0.7, color="tomato", linestyle="--", linewidth=1.5, label="p=0.7 threshold")
    ax.set_title(
        f"{role.capitalize()}\nmean={mean_p:.2f}  ≥0.7: {frac_conf:.0%}",
        fontsize=10,
    )
    ax.set_xlabel("Primary cluster probability")
    ax.set_ylabel("Players")
    ax.legend(fontsize=8)

fig.suptitle(f"GMM Soft Assignment Confidence — {YEAR}", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Players with the most ambiguous (borderline) cluster assignments
ROLE_INSPECT = "batter"  # change to "batter" or "catcher"

df = archetypes.get(ROLE_INSPECT)
if df is not None and "prob_primary" in df.columns:
    primary_label = df["cluster_id"].map(lambda c: label_for(ROLE_INSPECT, c))
    secondary_label = df["cluster_id_secondary"].map(lambda c: label_for(ROLE_INSPECT, c))
    view = df[["player_name", "cluster_id", "cluster_id_secondary", "prob_primary", "prob_secondary"]].copy()
    view["primary_label"] = primary_label
    view["secondary_label"] = secondary_label
    print(f"--- Most borderline {ROLE_INSPECT}s (lowest prob_primary) ---")
    display(view.nsmallest(15, "prob_primary")[["player_name", "primary_label", "secondary_label", "prob_primary", "prob_secondary"]].reset_index(drop=True))
    print(f"\n--- Most confident {ROLE_INSPECT}s (highest prob_primary) ---")
    display(view.nlargest(15, "prob_primary")[["player_name", "primary_label", "prob_primary"]].reset_index(drop=True))

## 6. PCA 2D Scatter

Project players into the first two principal components using the saved `archetype_clustering.joblib` model.

In [ ]:
import io

ROLE_PCA = "pitcher"  # change to "batter" or "catcher"


def load_joblib_from_s3(role: str) -> object | None:
    path = s3_path(role, "archetype_clustering.joblib")
    try:
        with fs.open(path, "rb") as f:
            return joblib.load(io.BytesIO(f.read()))
    except FileNotFoundError:
        print(f"[missing] {path}")
        return None


bundle = load_joblib_from_s3(ROLE_PCA)
df_arc = archetypes.get(ROLE_PCA)

if bundle and df_arc is not None:
    scaler = bundle["scaler"]
    pca = bundle["pca"]
    feats = bundle["feature_columns"]

    X = df_arc[feats].values
    X_scaled = scaler.transform(X)
    X_pca = pca.transform(X_scaled)

    cluster_ids = df_arc["cluster_id"].values
    palette = sns.color_palette("tab10", n_colors=len(np.unique(cluster_ids)))

    fig, ax = plt.subplots(figsize=(9, 7))
    for cid in sorted(np.unique(cluster_ids)):
        mask = cluster_ids == cid
        ax.scatter(
            X_pca[mask, 0], X_pca[mask, 1],
            label=label_for(ROLE_PCA, cid),
            alpha=0.7, s=40, color=palette[cid],
        )
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
    ax.set_title(f"{ROLE_PCA.capitalize()} PCA Space — {YEAR}", fontsize=13)
    ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.show()

    var_explained = pca.explained_variance_ratio_
    print(f"\nPCA: {len(var_explained)} components, {var_explained.sum():.1%} total variance explained")
    print("Per-component:", ", ".join(f"PC{i+1}={v:.1%}" for i, v in enumerate(var_explained[:6])))

## 7. Player Lookup

Find a player by name and inspect their cluster assignment, soft probabilities, and where they sit in PCA space.

In [ ]:
PLAYER_QUERY = "Guerrero"  # partial name, case-insensitive
ROLE_LOOKUP = "batter"     # "batter", "pitcher", or "catcher"

df = archetypes.get(ROLE_LOOKUP)
if df is not None:
    hits = df[df["player_name"].str.contains(PLAYER_QUERY, case=False, na=False)]
    if hits.empty:
        print(f"No {ROLE_LOOKUP} matching '{PLAYER_QUERY}'")
    else:
        prob_cols = sorted([c for c in hits.columns if c.startswith("prob_") and c not in ("prob_primary", "prob_secondary")])
        for _, row in hits.iterrows():
            print(f"Player  : {row['player_name']}  (id={row['player_id']})")
            print(f"Year    : {row['year']}   Role: {row['role']}   Position: {row.get('primary_position', '?')}")
            print(f"Cluster : {row['cluster_id']} — {label_for(ROLE_LOOKUP, row['cluster_id'])}")
            print(f"  prob_primary  = {row['prob_primary']:.3f}   ({label_for(ROLE_LOOKUP, row['cluster_id'])})")
            print(f"  prob_secondary= {row['prob_secondary']:.3f}   ({label_for(ROLE_LOOKUP, row['cluster_id_secondary'])})")
            if prob_cols:
                print("  All cluster probabilities:")
                for pc in prob_cols:
                    cid = int(pc.split("_")[1])
                    bar = "█" * int(row[pc] * 30)
                    print(f"    {pc} ({label_for(ROLE_LOOKUP, cid):<30s}): {row[pc]:.3f}  {bar}")
            print()

## 8. Player Similarity Explorer

In [ ]:
SIMILAR_PLAYER = "Guerrero"  # partial name
ROLE_SIM = "batter"          # "batter", "pitcher", or "catcher"

df_arc = archetypes.get(ROLE_SIM)
df_nbr = neighbors.get(ROLE_SIM)

if df_arc is not None and df_nbr is not None:
    matches = df_arc[df_arc["player_name"].str.contains(SIMILAR_PLAYER, case=False, na=False)]
    if matches.empty:
        print(f"No {ROLE_SIM} matching '{SIMILAR_PLAYER}'")
    else:
        # Use first match
        player_row = matches.iloc[0]
        pid = player_row["player_id"]
        print(f"Querying: {player_row['player_name']}  (id={pid})  "
              f"Cluster: {label_for(ROLE_SIM, player_row['cluster_id'])}\n")

        nbrs = df_nbr[df_nbr["player_id"] == pid].sort_values("neighbor_rank")

        # Enrich with neighbor's cluster info
        cid_map = df_arc.set_index("player_id")["cluster_id"].to_dict()
        nbrs = nbrs.copy()
        nbrs["neighbor_cluster"] = nbrs["neighbor_player_id"].map(cid_map)
        nbrs["neighbor_archetype"] = nbrs["neighbor_cluster"].map(
            lambda c: label_for(ROLE_SIM, int(c)) if pd.notna(c) else "?"
        )
        nbrs["same_cluster"] = nbrs["neighbor_cluster"] == player_row["cluster_id"]

        display(nbrs[["neighbor_rank", "neighbor_player_name", "neighbor_archetype", "distance", "same_cluster"]].reset_index(drop=True))

In [ ]:
# Pivot to wide format: one row per player, columns = rank_1 ... rank_k
ROLE_WIDE = "pitcher"  # change as needed
df_arc = archetypes.get(ROLE_WIDE)
df_nbr = neighbors.get(ROLE_WIDE)

if df_arc is not None and df_nbr is not None:
    pivot = (
        df_nbr
        .pivot(index="player_id", columns="neighbor_rank", values="neighbor_player_name")
        .rename(columns=lambda r: f"similar_{r}")
    )
    dist_pivot = (
        df_nbr
        .pivot(index="player_id", columns="neighbor_rank", values="distance")
        .rename(columns=lambda r: f"dist_{r}")
    )
    name_map = df_arc.set_index("player_id")["player_name"]
    cluster_map = df_arc.set_index("player_id")["cluster_id"].map(lambda c: label_for(ROLE_WIDE, c))

    wide = pd.concat([name_map.rename("player_name"), cluster_map.rename("archetype"), pivot, dist_pivot], axis=1)
    print(f"Wide similarity table: {wide.shape}")
    display(wide.head(10))

## 9. Distance Distributions

In [ ]:
fig, axes = plt.subplots(1, len(ROLES), figsize=(14, 4), sharey=False)
for ax, role in zip(axes, ROLES):
    df_nbr = neighbors.get(role)
    if df_nbr is None:
        ax.set_title(f"{role} (no data)")
        continue
    for rank, grp in df_nbr.groupby("neighbor_rank"):
        ax.hist(grp["distance"], bins=25, alpha=0.55, label=f"rank {rank}")
    ax.set_title(f"{role.capitalize()}", fontsize=11)
    ax.set_xlabel("PCA Distance")
    ax.set_ylabel("Players")
    ax.legend(fontsize=7)

fig.suptitle(f"KNN Distance Distribution by Rank — {YEAR}", fontsize=13)
plt.tight_layout()
plt.show()

## 10. Cross-Cluster Neighbor Analysis

For each cluster, what fraction of each player's top-K neighbors are in the **same** cluster?  
High same-cluster rate → tight, well-separated archetypes.

In [ ]:
for role in ROLES:
    df_arc = archetypes.get(role)
    df_nbr = neighbors.get(role)
    if df_arc is None or df_nbr is None:
        continue

    cid_map = df_arc.set_index("player_id")["cluster_id"]

    df_nbr = df_nbr.copy()
    df_nbr["query_cluster"] = df_nbr["player_id"].map(cid_map)
    df_nbr["neighbor_cluster"] = df_nbr["neighbor_player_id"].map(cid_map)
    df_nbr["same_cluster"] = df_nbr["query_cluster"] == df_nbr["neighbor_cluster"]

    # Same-cluster rate per query cluster
    rate = (
        df_nbr
        .groupby("query_cluster")["same_cluster"]
        .mean()
        .rename("same_cluster_rate")
        .to_frame()
    )
    rate["archetype"] = rate.index.map(lambda c: label_for(role, c))
    rate["n_players"] = df_arc.groupby("cluster_id").size()

    print(f"=== {role.upper()} ===")
    display(rate[["archetype", "n_players", "same_cluster_rate"]].sort_values("same_cluster_rate", ascending=False))
    overall = df_nbr["same_cluster"].mean()
    print(f"  Overall same-cluster rate: {overall:.1%}\n")

In [ ]:
# Cluster-to-cluster neighbor flow heatmap
ROLE_FLOW = "pitcher"
df_arc = archetypes.get(ROLE_FLOW)
df_nbr = neighbors.get(ROLE_FLOW)

if df_arc is not None and df_nbr is not None:
    cid_map = df_arc.set_index("player_id")["cluster_id"]
    df_nbr = df_nbr.copy()
    df_nbr["query_cluster"] = df_nbr["player_id"].map(cid_map)
    df_nbr["neighbor_cluster"] = df_nbr["neighbor_player_id"].map(cid_map)

    flow = (
        df_nbr
        .groupby(["query_cluster", "neighbor_cluster"])
        .size()
        .unstack(fill_value=0)
        .pipe(lambda m: m.div(m.sum(axis=1), axis=0))  # normalize rows
    )
    n_clusters = clustering_meta[ROLE_FLOW].get("n_clusters", flow.shape[0]) if clustering_meta[ROLE_FLOW] else flow.shape[0]
    tick_labels = [label_for(ROLE_FLOW, i) for i in range(n_clusters)]

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        flow,
        ax=ax,
        cmap="YlOrRd",
        annot=True,
        fmt=".0%",
        xticklabels=tick_labels,
        yticklabels=tick_labels,
        linewidths=0.5,
        cbar_kws={"label": "fraction of neighbors"},
    )
    ax.set_title(f"{ROLE_FLOW.capitalize()} — Neighbor Cluster Flow (row = query cluster)", fontsize=11)
    ax.set_xlabel("Neighbor cluster")
    ax.set_ylabel("Query cluster")
    plt.xticks(rotation=30, ha="right", fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    plt.show()

## 11. Clustering Quality Metrics

In [ ]:
rows = []
for role in ROLES:
    meta = clustering_meta.get(role)
    if not meta:
        continue
    rows.append({
        "role": role,
        "k": meta.get("n_clusters"),
        "pca_components": meta.get("pca_n_components"),
        "pca_variance": meta.get("pca_total_explained_variance"),
        "silhouette": meta.get("silhouette_score"),
        "davies_bouldin": meta.get("davies_bouldin_score"),
        "gmm_bic": meta.get("gmm_bic"),
        "gmm_aic": meta.get("gmm_aic"),
        "mean_max_prob": meta.get("mean_max_prob"),
        "frac_confident": meta.get("frac_confident_p_ge_0_7"),
        "n_samples": meta.get("n_samples"),
        "cov_type": meta.get("gmm_covariance_type"),
    })

if rows:
    df_metrics = pd.DataFrame(rows).set_index("role")
    display(df_metrics)

In [ ]:
# BIC / AIC sweep (if BIC selection was used)
for role in ROLES:
    meta = clustering_meta.get(role)
    if not meta or not meta.get("bic_sweep"):
        continue
    sweep = pd.DataFrame(meta["bic_sweep"])
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(sweep["k"], sweep["bic"], marker="o", label="BIC")
    ax.plot(sweep["k"], sweep["aic"], marker="s", linestyle="--", label="AIC")
    ax.axvline(meta["n_clusters"], color="tomato", linestyle=":", label=f"selected k={meta['n_clusters']}")
    ax.set_title(f"{role.capitalize()} — BIC/AIC Sweep")
    ax.set_xlabel("k (clusters)")
    ax.legend()
    plt.tight_layout()
    plt.show()